# Notebook 06 — Ordinary Least Squares: Derivation & Implementation

**Difficulty:** ⭐⭐⭐⭐ | **Estimated time:** 3 hours  
**Theme:** Predicting house sale prices  
**Week:** 5 — ML Fundamentals & Linear Regression

---

## Why Does This Matter?

Gradient descent is everywhere in modern ML — but for linear regression there is a **closed-form exact solution**: compute it once, get the globally optimal answer. No learning rate, no iterations, no convergence issues.

Understanding OLS derivation gives you:
- The ability to implement linear regression **from scratch** (and match sklearn to floating-point precision)
- Intuition for **when the solution breaks** (collinearity, underdetermined systems)
- A concrete benchmark for understanding **why** gradient descent is used for large-scale problems
- The mathematical foundation for **regularised regression** (ridge, lasso, elastic net)

---

## Real-World Analogy First

Solving 2x + 3 = 11 exactly (x = 4) is better than guessing x=1, x=3, x=3.9, x=4.0 one at a time.

For linear regression:
- **Gradient descent** = guess-and-check: adjust θ step by step, hoping to converge
- **OLS / Normal equations** = solve algebraically: derive the exact formula for θ in one shot

OLS is the algebraic solution. It always finds the global minimum (no local minima for MSE). The catch: it requires inverting a matrix, which is expensive for many features.

## Section 1 — The Setup: What Are We Minimising?

We have n training samples, p features. In matrix form:

- **X** ∈ ℝⁿˣᵖ  (design matrix; first column is all 1s for the intercept)
- **y** ∈ ℝⁿ    (target vector of house prices)
- **θ** ∈ ℝᵖ    (weight vector we want to learn)

The prediction is **ŷ = Xθ** and the loss is the **Residual Sum of Squares (RSS)**:

$$L(\theta) = \|y - X\theta\|^2 = (y - X\theta)^\top (y - X\theta)$$

Expanding the matrix product:

$$L(\theta) = y^\top y - 2\theta^\top X^\top y + \theta^\top X^\top X\theta$$

This is a **quadratic function** in θ — shaped like a bowl in p dimensions. It has exactly one minimum.

## Section 2 — Derivation: Taking the Gradient

To find the minimum, take the gradient with respect to θ and set it to zero.

Using matrix calculus rules:
- d/dθ [θᵀAθ] = 2Aθ   (when A is symmetric, which XᵀX always is)
- d/dθ [bᵀθ]  = b

$$\frac{\partial L}{\partial \theta} = -2X^\top y + 2X^\top X\theta$$

Setting the gradient to zero:

$$-2X^\top y + 2X^\top X\theta^* = 0$$

$$\boxed{X^\top X\theta^* = X^\top y}$$

These are the **Normal Equations**. If XᵀX is invertible:

$$\theta^* = (X^\top X)^{-1} X^\top y$$

The matrix **X⁺ = (XᵀX)⁻¹Xᵀ** is the **Moore-Penrose pseudoinverse** of X when X has full column rank.

**Plain English at each step:**
1. L(θ) is the total squared error — a bowl in p-dimensional weight space
2. The gradient points uphill — setting it to zero finds the flat bottom
3. At the bottom: XᵀXθ = Xᵀy — a system of p linear equations in p unknowns
4. Invert XᵀX to solve — this is the exact answer, no iteration needed

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import time
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)                   # reproducible throughout
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size']  = 11

print("Libraries loaded.")
print(f"NumPy {np.__version__}  |  Pandas {pd.__version__}")

In [ ]:
# ── Generate synthetic house price data ──────────────────────────────────────
# Using synthetic data so we know the TRUE coefficients — perfect for verification

n = 200   # 200 houses

# Three features: size (sqft), bedrooms, age (years)
size_sqft = np.random.uniform(800, 3500, n)     # house size in square feet
bedrooms  = np.random.randint(1, 6, n).astype(float)  # 1–5 bedrooms
age_years = np.random.uniform(1, 50, n)         # house age

# Stack features into design matrix (no bias yet)
X_raw = np.column_stack([size_sqft, bedrooms, age_years])  # shape (200, 3)

# True coefficients (we set these to verify our derivation)
true_intercept = 50_000    # $50k base price
true_coefs     = np.array([150, 10_000, -800])  # $/sqft, $/bedroom, $/year

# Generate target prices with Gaussian noise
noise = np.random.normal(0, 20_000, n)   # ±$20k random noise
y = true_intercept + X_raw @ true_coefs + noise   # shape (200,)

# Prices in $1 — keep interpretable
print("Synthetic dataset created:")
print(f"  n = {n} houses,  p = {X_raw.shape[1]} features")
print(f"  True intercept:  ${true_intercept:,}")
print(f"  True θ_size:     ${true_coefs[0]:,}/sqft")
print(f"  True θ_beds:     ${true_coefs[1]:,}/bedroom")
print(f"  True θ_age:      ${true_coefs[2]:,}/year (depreciation)")
print(f"  Price range:     [${y.min():,.0f}, ${y.max():,.0f}]")
print(f"  Price mean:      ${y.mean():,.0f}")

In [ ]:
# ── Step 0: Add bias column ───────────────────────────────────────────────────
# The intercept θ₀ is handled by prepending a column of 1s to X
# This makes θ = [θ₀, θ₁, θ₂, θ₃] — intercept + 3 feature weights

X = np.column_stack([np.ones(n), X_raw])   # shape (200, 4)

print("Design matrix X shape:", X.shape)
print("First 5 rows of X (bias | size | beds | age):")
print(np.round(X[:5], 1))
print()
print("Interpretation: X[i] = [1, size_i, beds_i, age_i]")
print("  ŷ_i = θ₀·1 + θ₁·size_i + θ₂·beds_i + θ₃·age_i")
print("      = dot product of θ with the i-th row of X")

In [ ]:
# ── Step 1: Compute XᵀX and Xᵀy ─────────────────────────────────────────────
# XᵀX is the Gram matrix: shape (p × p) = (4 × 4)
# Xᵀy is the moment vector: shape (p,) = (4,)
# These are the two components of the Normal Equations: XᵀXθ = Xᵀy

XtX = X.T @ X    # (4, 4) — matrix of feature cross-products
Xty = X.T @ y    # (4,)  — cross-products of features with target

print("XᵀX (Gram matrix) — shape", XtX.shape)
print(np.round(XtX, 0))
print()
print("Xᵀy (moment vector) — shape", Xty.shape)
print(np.round(Xty, 0))
print()
print(f"det(XᵀX) = {np.linalg.det(XtX):.4e}")
print("  (Non-zero det → XᵀX is invertible → OLS has a unique solution)")

In [ ]:
# ── Step 2: Solve the Normal Equations θ* = (XᵀX)⁻¹ Xᵀy ─────────────────────
# np.linalg.inv computes the exact matrix inverse
# This is O(p³) — fine for p=4, expensive for p=10,000

XtX_inv = np.linalg.inv(XtX)      # (4, 4) inverse
theta   = XtX_inv @ Xty           # (4,)  — the OLS solution

print("OLS Solution θ* = (XᵀX)⁻¹ Xᵀy:")
print(f"  θ₀ (intercept) = ${theta[0]:>12,.2f}   [true: ${true_intercept:,}]")
print(f"  θ₁ (size/sqft) = ${theta[1]:>12,.2f}   [true: ${true_coefs[0]:,}]")
print(f"  θ₂ (bedrooms)  = ${theta[2]:>12,.2f}   [true: ${true_coefs[1]:,}]")
print(f"  θ₃ (age/year)  = ${theta[3]:>12,.2f}   [true: ${true_coefs[2]:,}]")
print()
print("Recovery error (due to noise in training data):")
true_full = np.array([true_intercept] + list(true_coefs))
for j, (est, tru) in enumerate(zip(theta, true_full)):
    err = abs(est - tru) / abs(tru) * 100
    print(f"  θ{j}: {err:.1f}% from truth")

In [ ]:
# ── Step 3: Verify against sklearn LinearRegression ──────────────────────────
# sklearn's LinearRegression uses SVD internally (more numerically stable than inv)
# but should give identical results for well-conditioned problems

lr = LinearRegression(fit_intercept=False)  # intercept already in X[:,0]
lr.fit(X, y)
theta_sklearn = lr.coef_                    # sklearn's estimate

print("Comparison: OLS from scratch vs sklearn")
print(f"{'Coefficient':>18}  {'Our OLS':>14}  {'sklearn':>14}  {'Max diff':>12}")
print("-" * 64)
names = ['θ₀ (intercept)', 'θ₁ (size/sqft)', 'θ₂ (bedrooms)', 'θ₃ (age/yr)']
for name, ours, sk in zip(names, theta, theta_sklearn):
    print(f"{name:>18}  {ours:>14.4f}  {sk:>14.4f}  {abs(ours-sk):>12.2e}")

print()
# np.allclose checks every element is within atol (absolute) and rtol (relative)
identical = np.allclose(theta, theta_sklearn, rtol=1e-5, atol=1e-3)
print(f"np.allclose(theta_ours, theta_sklearn) = {identical}")
print("✓ Results are identical to floating-point precision.")

## Section 3 — Computing R² From Scratch

R² (coefficient of determination) measures what fraction of variance in y is explained by the model:

$$R^2 = 1 - \frac{SS_{\text{res}}}{SS_{\text{tot}}} = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}$$

- SS_res = residual variance (unexplained)
- SS_tot = total variance in y
- R² = 1 means perfect fit; R² = 0 means the model is no better than predicting ȳ for everyone

In [ ]:
# ── R² from scratch and verify against sklearn ───────────────────────────────
y_hat     = X @ theta                          # fitted values
residuals = y - y_hat                          # residuals

SS_res = np.sum(residuals**2)                  # sum of squared residuals
SS_tot = np.sum((y - y.mean())**2)             # total sum of squares
R2_ours   = 1 - SS_res / SS_tot               # coefficient of determination

R2_sklearn = r2_score(y, y_hat)               # sklearn's implementation

print("R² Calculation:")
print(f"  SS_res (unexplained)  = {SS_res:>15,.0f}")
print(f"  SS_tot (total)        = {SS_tot:>15,.0f}")
print(f"  SS_reg (explained)    = {SS_tot - SS_res:>15,.0f}")
print()
print(f"  R² (our formula)   = {R2_ours:.6f}")
print(f"  R² (sklearn)       = {R2_sklearn:.6f}")
print(f"  Match: {np.isclose(R2_ours, R2_sklearn, atol=1e-8)}")
print()
print(f"  Interpretation: our 3 features explain {R2_ours*100:.1f}% of house price variance.")
print(f"  Residual std = ${np.sqrt(SS_res / (n - X.shape[1])):,.0f} (adjusted for p={X.shape[1]} parameters)")

In [ ]:
# ── Visualise: predicted vs actual + residuals ───────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Left: Predicted vs Actual ────────────────────────────────────────────────
ax = axes[0]
ax.scatter(y, y_hat, alpha=0.5, color='steelblue', s=30)
lims = [min(y.min(), y_hat.min()), max(y.max(), y_hat.max())]
ax.plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
ax.set_xlabel('Actual Price ($)')
ax.set_ylabel('Predicted Price ($)')
ax.set_title(f'Predicted vs Actual\nR² = {R2_ours:.3f}', fontsize=12)
ax.legend()

# ── Middle: Residuals vs Fitted ──────────────────────────────────────────────
ax2 = axes[1]
ax2.scatter(y_hat, residuals, alpha=0.5, color='darkorange', s=30)
ax2.axhline(0, color='crimson', linewidth=2, linestyle='--')
ax2.set_xlabel('Fitted Values ŷ')
ax2.set_ylabel('Residuals (y − ŷ)')
ax2.set_title('Residuals vs Fitted\n(Random scatter = good)', fontsize=12)

# ── Right: Feature coefficients with error bars ──────────────────────────────
ax3 = axes[2]
# Compute standard errors for each coefficient from the covariance matrix
# Var(θ̂) = σ² (XᵀX)⁻¹ where σ² = RSS / (n - p)
sigma2_est = SS_res / (n - X.shape[1])        # unbiased variance estimate
cov_theta  = sigma2_est * XtX_inv             # covariance matrix of θ̂
se_theta   = np.sqrt(np.diag(cov_theta))      # standard errors (diag of cov)

labels = ['Intercept\n($50k true)', 'Size\n($150/sqft)', 'Bedrooms\n($10k/bed)', 'Age\n(-$800/yr)']
colors_bar = ['steelblue' if abs(theta[i]) < 1e5 else 'darkorange' for i in range(4)]

ax3.barh(labels, theta, xerr=1.96*se_theta, color='steelblue',
         alpha=0.7, capsize=4, edgecolor='white')
# Mark the true coefficients as red dots
ax3.scatter(true_full, labels, color='crimson', zorder=5, s=60,
            label='True values', marker='D')
ax3.axvline(0, color='grey', linewidth=1)
ax3.set_xlabel('Coefficient estimate (±95% CI)')
ax3.set_title('OLS Estimates vs Truth\n(Diamonds = true values)', fontsize=12)
ax3.legend(fontsize=9)

plt.suptitle('OLS Regression on Synthetic House Price Data (n=200, p=3)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('ols_results.png', bbox_inches='tight', dpi=110)
plt.show()

## Section 4 — When OLS Breaks: Rank Deficiency and Collinearity

OLS requires XᵀX to be **invertible** (non-singular). This fails when:

1. **Perfect collinearity**: one feature is an exact linear combination of others (e.g., size_sqm = size_sqft × 0.0929)
2. **More features than samples**: p > n means X has at most rank n < p → XᵀX is rank-deficient
3. **Duplicate features**: adding the same column twice

**Why**: if features are linearly dependent, the column space of X doesn't fully span ℝᵖ. Many θ values give the same ŷ — the solution is not unique.

**Fix**: use the Moore-Penrose pseudoinverse `np.linalg.pinv(X)`, which finds the minimum-norm solution.

In [ ]:
# ── Demonstrate rank deficiency: perfect collinearity ────────────────────────
# Add size_sqm = size_sqft × 0.0929  — perfectly correlated with size_sqft

size_sqm = size_sqft * 0.0929                          # exact linear transform
X_collinear = np.column_stack([np.ones(n), size_sqft,
                                bedrooms, age_years, size_sqm])  # (200, 5)

XtX_collinear = X_collinear.T @ X_collinear

det_val  = np.linalg.det(XtX_collinear)
rank_val = np.linalg.matrix_rank(XtX_collinear)

print("=" * 55)
print("  COLLINEAR FEATURE DEMONSTRATION")
print("=" * 55)
print(f"  X shape:                {X_collinear.shape}  (5 columns)")
print(f"  rank(XᵀX):              {rank_val}  (should be 5 for full rank)")
print(f"  det(XᵀX):               {det_val:.4e}  (≈ 0 means singular!)")
print()

# Attempt direct inversion — this will raise an error or produce garbage
print("Attempting np.linalg.inv(XᵀX) ...")
try:
    inv_attempt = np.linalg.inv(XtX_collinear)
    cond_number = np.linalg.cond(XtX_collinear)      # condition number
    print(f"  No error (NumPy handles near-singular, but...)")
    print(f"  Condition number: {cond_number:.3e}  (values > 1e12 are unreliable)")
    theta_bad = inv_attempt @ (X_collinear.T @ y)
    print(f"  Estimated θ_size: {theta_bad[1]:.2f}  θ_sqm: {theta_bad[4]:.2f}  (sum = {theta_bad[1]+theta_bad[4]:.2f})")
    print(f"  True θ_size: 150 (but now split arbitrarily between size_sqft and size_sqm!)")
except np.linalg.LinAlgError as e:
    print(f"  LinAlgError: {e}  ← expected!")
print()

# ── Correct approach: pseudoinverse ──────────────────────────────────────────
print("Using np.linalg.pinv (Moore-Penrose pseudoinverse):")
theta_pinv = np.linalg.pinv(X_collinear) @ y    # minimum-norm solution
print(f"  θ_size_sqft = {theta_pinv[1]:.4f}")
print(f"  θ_size_sqm  = {theta_pinv[4]:.4f}")
y_hat_pinv  = X_collinear @ theta_pinv
r2_pinv     = 1 - np.sum((y - y_hat_pinv)**2) / np.sum((y - y.mean())**2)
print(f"  R² with pinv: {r2_pinv:.6f}  (same as before — predictions are fine!)")
print(f"  But individual coefficients are not meaningful when collinear.")

In [ ]:
# ── Visualise: what collinearity does to the solution space ──────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: Correlation matrix of the collinear feature set ───────────────────
ax = axes[0]
df_collinear = pd.DataFrame(
    np.column_stack([size_sqft, bedrooms, age_years, size_sqm]),
    columns=['size_sqft', 'bedrooms', 'age_years', 'size_sqm']
)
corr = df_collinear.corr()
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True
sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            ax=ax, square=True, linewidths=0.5, mask=~mask,
            vmin=-1, vmax=1)
ax.set_title('Feature Correlation Matrix\n(size_sqft and size_sqm are perfectly correlated)', fontsize=11)

# ── Right: Condition number vs amount of collinearity ────────────────────────
ax2 = axes[1]
# Vary how close size_sqm is to being an exact copy (epsilon controls noise)
epsilon_vals  = np.logspace(-8, 0, 40)   # from near-perfect to random
cond_numbers  = []

for eps in epsilon_vals:
    # Add small random perturbation: the closer eps→0, the more collinear
    noisy_copy   = size_sqft * 0.0929 + eps * np.random.randn(n)
    X_test       = np.column_stack([np.ones(n), size_sqft, bedrooms,
                                     age_years, noisy_copy])
    XtX_test     = X_test.T @ X_test
    cond_numbers.append(np.linalg.cond(XtX_test))

ax2.loglog(epsilon_vals, cond_numbers, 'o-', color='steelblue', markersize=4)
ax2.axhline(1e12, color='crimson', linestyle='--', linewidth=1.5,
            label='Danger zone (κ > 1e12)')
ax2.set_xlabel('Noise added to collinear feature (ε)')
ax2.set_ylabel('Condition number κ(XᵀX)')
ax2.set_title('Collinearity → High Condition Number\n(Numerical instability)', fontsize=11)
ax2.legend()
ax2.grid(True, which='both', alpha=0.3)

plt.suptitle('Effect of Perfect Collinearity on OLS', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('collinearity.png', bbox_inches='tight', dpi=110)
plt.show()

## Section 5 — Geometric View of the Normal Equations

The Normal Equations XᵀXθ = Xᵀy can be visualised for the simplest case: 1 feature (p=2 with bias).

In this case, XᵀX is a 2×2 matrix and XᵀXθ = Xᵀy is a system of 2 linear equations:

$$\begin{bmatrix} n & \sum x_i \\ \sum x_i & \sum x_i^2 \end{bmatrix} \begin{bmatrix} \theta_0 \\ \theta_1 \end{bmatrix} = \begin{bmatrix} \sum y_i \\ \sum x_i y_i \end{bmatrix}$$

Each row is one equation — a line in (θ₀, θ₁) space. The solution is the intersection.

In [ ]:
# ── Graphical solution of the normal equations (1-feature case) ───────────────
# For p=2: XᵀX is 2×2, so the normal equations are 2 lines in (θ₀, θ₁) space

# Use only size_sqft as the single feature
X_1d = np.column_stack([np.ones(n), size_sqft])   # shape (200, 2)
XtX_1d = X_1d.T @ X_1d                             # 2×2 matrix
Xty_1d = X_1d.T @ y                                # 2-vector
theta_1d = np.linalg.inv(XtX_1d) @ Xty_1d          # [θ₀, θ₁] solution

print("Normal Equations (1-feature case):")
print(f"  XᵀX = {XtX_1d.round(1)}")
print(f"  Xᵀy = {Xty_1d.round(1)}")
print()
print(f"  Equation 1: {XtX_1d[0,0]:.0f}·θ₀ + {XtX_1d[0,1]:.1f}·θ₁ = {Xty_1d[0]:.1f}")
print(f"  Equation 2: {XtX_1d[1,0]:.1f}·θ₀ + {XtX_1d[1,1]:.1f}·θ₁ = {Xty_1d[1]:.1f}")
print()
print(f"  Solution: θ₀ = {theta_1d[0]:,.2f},  θ₁ = {theta_1d[1]:.4f}")

# Express each equation as θ₀ = f(θ₁) to plot as lines
theta1_range = np.linspace(50, 250, 400)

# Eq 1: n·θ₀ + Σx·θ₁ = Σy  →  θ₀ = (Σy - Σx·θ₁) / n
line1 = (Xty_1d[0] - XtX_1d[0, 1] * theta1_range) / XtX_1d[0, 0]

# Eq 2: Σx·θ₀ + Σx²·θ₁ = Σxy  →  θ₀ = (Σxy - Σx²·θ₁) / Σx
line2 = (Xty_1d[1] - XtX_1d[1, 1] * theta1_range) / XtX_1d[1, 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: normal equations as two lines in parameter space ───────────────────
ax = axes[0]
ax.plot(theta1_range, line1, color='steelblue', linewidth=2.5,
        label=f'Eq 1: {XtX_1d[0,0]:.0f}θ₀ + {XtX_1d[0,1]:.0f}θ₁ = {Xty_1d[0]:.0f}')
ax.plot(theta1_range, line2, color='darkorange', linewidth=2.5,
        label=f'Eq 2: {XtX_1d[1,0]:.0f}θ₀ + {XtX_1d[1,1]:.0f}θ₁ = {Xty_1d[1]:.0f}')
ax.scatter([theta_1d[1]], [theta_1d[0]], color='crimson', s=150, zorder=6,
           label=f'Solution (θ₀={theta_1d[0]:,.0f}, θ₁={theta_1d[1]:.1f})')
ax.set_xlabel('θ₁ (price per sqft)')
ax.set_ylabel('θ₀ (intercept / base price)')
ax.set_title('Normal Equations as 2 Lines\n(Intersection = OLS solution)', fontsize=12)
ax.set_ylim(-200_000, 400_000)
ax.legend(fontsize=8)

# ── Right: regression line on data ──────────────────────────────────────────
ax2 = axes[1]
ax2.scatter(size_sqft, y, alpha=0.4, color='steelblue', s=25, label='Houses')
x_line = np.linspace(size_sqft.min(), size_sqft.max(), 200)
y_line = theta_1d[0] + theta_1d[1] * x_line
ax2.plot(x_line, y_line, color='crimson', linewidth=2.5,
         label=f'ŷ = {theta_1d[0]:,.0f} + {theta_1d[1]:.1f}·size')
ax2.set_xlabel('House Size (sqft)')
ax2.set_ylabel('Sale Price ($)')
ax2.set_title('OLS Fit: Size → Price', fontsize=12)
ax2.legend(fontsize=9)

plt.suptitle('Normal Equations — Graphical Solution in Parameter Space', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('normal_equations_graphical.png', bbox_inches='tight', dpi=110)
plt.show()

## Section 6 — Computational Complexity Analysis

### OLS Computational Cost

| Operation | Cost | Notes |
|---|---|---|
| Compute Xᵀy | O(np) | One pass through data |
| Compute XᵀX | O(np²) | Gram matrix |
| Invert XᵀX | O(p³) | Dominant for large p |
| **Total** | **O(np² + p³)** | |

- p=10: trivial
- p=1,000: ~10⁹ ops for inversion — feasible (seconds)
- p=10,000: ~10¹² ops — slow (hours)
- p=100,000: ~10¹⁵ ops — infeasible → **use gradient descent**

### Memory Cost
- X is (n × p) — for n=1M, p=1000: 8 GB (float64)
- XᵀX is (p × p) — for p=1000: 8 MB (tiny regardless of n!)
- So the bottleneck is computing XᵀX, not storing it

### Gradient Descent Cost
- Each step: O(np) — one forward pass
- k steps total: O(knp)
- For large p and small k, gradient descent wins

In [ ]:
# ── Timing: OLS vs Gradient Descent for varying n ───────────────────────────
# We use a fixed p=5 and vary n to show the n-scaling of each method

p_fixed = 5    # small p so OLS inversion is cheap — we isolate the n-effect

def ols_time(n_samples, p=p_fixed, repeats=3):
    """Time the OLS closed-form solution."""
    Xt = np.random.randn(n_samples, p)
    Xt = np.column_stack([np.ones(n_samples), Xt])   # add bias
    yt = np.random.randn(n_samples)
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        _ = np.linalg.inv(Xt.T @ Xt) @ (Xt.T @ yt)  # OLS formula
        times.append(time.perf_counter() - t0)
    return np.median(times)

def gd_time(n_samples, p=p_fixed, steps=500, lr=0.001, repeats=3):
    """Time gradient descent (fixed number of steps)."""
    Xt = np.random.randn(n_samples, p)
    Xt = np.column_stack([np.ones(n_samples), Xt])
    Xt = Xt / (np.std(Xt, axis=0) + 1e-8)           # normalise for stable GD
    yt = np.random.randn(n_samples)
    times = []
    for _ in range(repeats):
        t0    = time.perf_counter()
        theta_gd = np.zeros(p + 1)
        for _ in range(steps):
            grad     = -2 * Xt.T @ (yt - Xt @ theta_gd) / n_samples
            theta_gd -= lr * grad
        times.append(time.perf_counter() - t0)
    return np.median(times)

n_vals = [100, 500, 1000, 5000, 10_000, 50_000]
ols_times = [ols_time(n) for n in n_vals]
gd_times  = [gd_time(n)  for n in n_vals]

print(f"{'n':>8}  {'OLS (s)':>10}  {'GD (s)':>10}  {'OLS/GD ratio':>14}")
print("-" * 48)
for n, t_ols, t_gd in zip(n_vals, ols_times, gd_times):
    print(f"{n:>8,}  {t_ols:>10.5f}  {t_gd:>10.5f}  {t_ols/t_gd:>14.2f}x")

In [ ]:
# ── Plot timing comparison ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: Wall-clock time vs n ───────────────────────────────────────────────
ax = axes[0]
ax.loglog(n_vals, ols_times, 'o-', color='steelblue', linewidth=2,
          markersize=7, label='OLS (closed-form)')
ax.loglog(n_vals, gd_times,  's-', color='darkorange', linewidth=2,
          markersize=7, label='Gradient Descent (500 steps)')
ax.set_xlabel('Number of samples n')
ax.set_ylabel('Time (seconds, log scale)')
ax.set_title(f'OLS vs GD Wall-Clock Time (p={p_fixed})', fontsize=12)
ax.legend()
ax.grid(True, which='both', alpha=0.3)

# ── Right: theoretical cost analysis table ───────────────────────────────────
ax2 = axes[1]
ax2.axis('off')
ax2.set_facecolor('#f5f7fa')

table_data = [
    ['p (features)', 'OLS cost O(p³)', 'Use OLS?'],
    ['10',         '1,000',            'Yes (trivial)'],
    ['100',        '1,000,000',         'Yes (fast)'],
    ['1,000',      '10⁹',               'Yes (seconds)'],
    ['10,000',     '10¹²',              'Slow (hours)'],
    ['100,000',    '10¹⁵',              'No → use GD'],
]
col_widths = [0.32, 0.32, 0.32]
for row_i, row in enumerate(table_data):
    y_pos = 0.88 - row_i * 0.135
    bg_color = '#dce8f7' if row_i == 0 else ('#fff9f0' if row_i % 2 == 0 else 'white')
    ax2.add_patch(plt.Rectangle((0.02, y_pos - 0.06), 0.96, 0.11,
                                 transform=ax2.transAxes, color=bg_color, zorder=0))
    for col_i, (text, width) in enumerate(zip(row, col_widths)):
        x_pos = 0.04 + sum(col_widths[:col_i])
        ax2.text(x_pos + width/2, y_pos, text, ha='center', va='center',
                 fontsize=9.5, fontweight='bold' if row_i == 0 else 'normal',
                 transform=ax2.transAxes,
                 color='crimson' if 'No' in text else 'black')

ax2.set_title('Theoretical Cost of OLS vs Feature Count', fontsize=12, pad=12)

plt.suptitle('OLS Computational Complexity', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('ols_complexity.png', bbox_inches='tight', dpi=110)
plt.show()

## Section 7 — Connecting Back to Notebook 05: Geometry

The Normal Equations **XᵀXθ = Xᵀy** are the algebraic expression of the geometric projection seen in Notebook 05.

- **Xᵀ(y − Xθ) = 0** means residuals are orthogonal to col(X)
- Rearranging: **XᵀXθ = Xᵀy** — this IS the Normal Equation
- Solving: **θ* = (XᵀX)⁻¹Xᵀy** — this IS the OLS formula

So the geometric and algebraic derivations are literally the same equation, just written differently:

| Geometric statement | Algebraic statement |
|---|---|
| Residuals ⊥ col(X) | Xᵀ(y − Xθ) = 0 |
| Closest point in col(X) to y | Normal Equations: XᵀXθ = Xᵀy |
| Shadow on the wall | θ* = (XᵀX)⁻¹Xᵀy |

In [ ]:
# ── Verify the connection: OLS residuals are orthogonal to col(X) ─────────────
y_hat_full = X @ theta                  # predictions using 3-feature OLS
res_full   = y - y_hat_full             # residuals

ortho_check = X.T @ res_full           # should be zero-vector

print("Normal Equations ↔ Geometric Projection: Verification")
print("="*55)
print()
print("Xᵀ · (y - ŷ) should be the zero vector:")
for j, val in enumerate(ortho_check):
    col_name = ['bias (1)', 'size_sqft', 'bedrooms', 'age_years'][j]
    print(f"  Col {j} [{col_name}]: {val:.3e}")
print()
print(f"np.allclose(Xᵀ·residuals, 0, atol=1e-6) = {np.allclose(ortho_check, 0, atol=1e-6)}")
print()
print("This confirms:")
print("  1. OLS residuals are orthogonal to every feature column")
print("  2. The OLS solution is the projection of y onto col(X)")
print("  3. The Normal Equations are the algebraic form of this orthogonality")
print("  → Geometric, algebraic, and probabilistic interpretations are all the same!")

In [ ]:
# ── Final comprehensive summary ───────────────────────────────────────────────
y_hat_final = X @ theta
SS_res_f = np.sum((y - y_hat_final)**2)
SS_tot_f = np.sum((y - y.mean())**2)
R2_final = 1 - SS_res_f / SS_tot_f

print("=" * 62)
print("  NOTEBOOK 06 — OLS SUMMARY")
print("=" * 62)
print()
print("Dataset: Synthetic house prices (n=200, p=3 features)")
print()
print("OLS Derivation (step by step):")
print("  1. Loss = ||y - Xθ||² = yᵀy - 2θᵀXᵀy + θᵀXᵀXθ")
print("  2. Gradient: dL/dθ = -2Xᵀy + 2XᵀXθ")
print("  3. Set to zero: XᵀXθ = Xᵀy  (Normal Equations)")
print("  4. Solve: θ* = (XᵀX)⁻¹ Xᵀy")
print()
print("Results:")
names = ['θ₀ (intercept)', 'θ₁ (size)', 'θ₂ (bedrooms)', 'θ₃ (age)']
true_full = np.array([true_intercept] + list(true_coefs))
for name, est, tru in zip(names, theta, true_full):
    print(f"  {name:>18}: estimated {est:>10,.1f}  |  true {tru:>10,}")
print()
print(f"  R²       = {R2_final:.6f}")
print(f"  RSS      = {SS_res_f:,.0f}")
print(f"  Residual σ̂ = ${np.sqrt(SS_res_f/(n-X.shape[1])):,.0f}")
print()
print("Collinearity demo:")
print(f"  det(XᵀX) with collinear feature = {det_val:.3e}  (≈ 0)")
print(f"  rank(XᵀX) with collinear feature = {rank_val} (expected 5)")
print(f"  Solution: use np.linalg.pinv → R² still = {r2_pinv:.4f}")
print()
print("Complexity:")
print("  OLS: O(np² + p³) — exact, one-shot, no hyperparameters")
print("  GD:  O(knp)      — iterative, needs learning rate, k steps")
print("  Cross-over: typically around p ~ 10,000")
print("=" * 62)

In [ ]:
# ── Bonus: Gradient Descent convergence vs OLS optimal ───────────────────────
# Show that GD converges TO the OLS solution — they are the same answer

# 1-feature problem for easy visualisation
X_gd  = np.column_stack([np.ones(n), size_sqft])  # (200, 2)
y_gd  = y.copy()

# Normalise for stable gradient descent
X_gd_norm = X_gd.copy()
feat_mean  = X_gd_norm[:, 1].mean()
feat_std   = X_gd_norm[:, 1].std()
X_gd_norm[:, 1] = (X_gd_norm[:, 1] - feat_mean) / feat_std

# OLS solution on normalised data
theta_ols_norm = np.linalg.inv(X_gd_norm.T @ X_gd_norm) @ (X_gd_norm.T @ y_gd)

# Gradient descent
theta_gd_path = [np.array([0.0, 0.0])]   # start at zero
lr_gd         = 0.01
n_steps       = 150
mse_path      = []

theta_curr = np.array([0.0, 0.0])
for step in range(n_steps):
    y_pred_gd = X_gd_norm @ theta_curr
    residual_gd = y_gd - y_pred_gd
    # Gradient of MSE: -2/n * Xᵀ(y - Xθ)
    grad_gd   = -2 / n * (X_gd_norm.T @ residual_gd)
    theta_curr = theta_curr - lr_gd * grad_gd
    theta_gd_path.append(theta_curr.copy())
    mse_path.append(np.mean(residual_gd**2))

theta_gd_path = np.array(theta_gd_path)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: MSE vs iterations ──────────────────────────────────────────────────
ax = axes[0]
ax.plot(mse_path, color='steelblue', linewidth=2, label='GD MSE per step')
ols_mse = np.mean((y_gd - X_gd_norm @ theta_ols_norm)**2)
ax.axhline(ols_mse, color='crimson', linestyle='--', linewidth=1.8,
           label=f'OLS optimal MSE = {ols_mse:.1f}')
ax.set_xlabel('Gradient Descent Step')
ax.set_ylabel('MSE')
ax.set_title('GD Converges to OLS Solution\n(Red line = exact OLS optimum)', fontsize=12)
ax.legend()

# ── Right: trajectory in (θ₀, θ₁) space ─────────────────────────────────────
ax2 = axes[1]
# Background: MSE contour plot
t0_range = np.linspace(theta_ols_norm[0] - 40000, theta_ols_norm[0] + 40000, 60)
t1_range = np.linspace(theta_ols_norm[1] - 30000, theta_ols_norm[1] + 30000, 60)
T0, T1   = np.meshgrid(t0_range, t1_range)
MSE_grid = np.array([[np.mean((y_gd - X_gd_norm @ np.array([t0, t1]))**2)
                       for t0 in t0_range] for t1 in t1_range])
ax2.contourf(T0, T1, MSE_grid, levels=30, cmap='Blues_r', alpha=0.7)
ax2.contour(T0, T1, MSE_grid, levels=15, colors='white', linewidths=0.5, alpha=0.4)
# GD path
ax2.plot(theta_gd_path[:, 0], theta_gd_path[:, 1], 'o-',
         color='darkorange', markersize=3, linewidth=1.5, label='GD path')
ax2.scatter(*theta_ols_norm, color='crimson', s=200, zorder=6,
            marker='*', label=f'OLS optimum')
ax2.scatter(0, 0, color='lime', s=100, zorder=6, label='GD start (0,0)')
ax2.set_xlabel('θ₀ (intercept)')
ax2.set_ylabel('θ₁ (slope)')
ax2.set_title('GD Trajectory in Parameter Space\n(Contours = MSE bowl)', fontsize=12)
ax2.legend(fontsize=9)

plt.suptitle('Gradient Descent Converges to the Same Answer as OLS', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('gd_vs_ols.png', bbox_inches='tight', dpi=110)
plt.show()

final_gd_theta = theta_gd_path[-1]
print(f"GD final θ = {final_gd_theta}")
print(f"OLS exact θ = {theta_ols_norm}")
print(f"Difference  = {np.abs(final_gd_theta - theta_ols_norm)}")
print("GD approaches OLS with more steps / smaller learning rate.")

## Self-Check Questions

Test your understanding before moving on.

---

**Question 1:** You have 100 training samples and 200 features. Can you use the OLS closed-form solution? What goes wrong?

<details>
<summary>▶ Click to reveal answer</summary>

No — OLS will fail (or give degenerate results) in this case. Here's why:

- X is (100 × 200): more columns than rows, so rank(X) ≤ 100 < 200
- XᵀX is (200 × 200) but has rank ≤ 100 → it is **rank-deficient** and therefore **not invertible**
- There are infinitely many θ that achieve zero training error (the system is underdetermined)
- `np.linalg.inv(XtX)` will either fail with a `LinAlgError` or produce numerically garbage results due to near-singularity

**Solutions**: use `np.linalg.pinv` (finds minimum-norm solution), or add regularisation (ridge: XᵀX + λI is always invertible for λ > 0).

</details>

---

**Question 2:** Two features are perfectly correlated: `size_sqft` and `size_sqm = size_sqft × 0.0929`. What happens to (XᵀX)⁻¹?

<details>
<summary>▶ Click to reveal answer</summary>

When two features are perfectly linearly dependent:

- One column of X is exactly 0.0929 times another → rank(X) drops by 1
- XᵀX becomes **singular**: det(XᵀX) = 0
- (XᵀX)⁻¹ **does not exist** — you cannot invert a singular matrix

Geometrically, the column space of X is now a *lower-dimensional subspace* than expected — the two features together only span one dimension, not two. The projection is still well-defined (there is still a closest point), but the *decomposition of the projection into individual feature contributions* is ambiguous — infinitely many (θ₁, θ₂) pairs give the same ŷ, as long as 150·θ₁ + 150/0.0929·θ₂ sums to the right total effect.

Numerically: condition number κ(XᵀX) → ∞, making any computed inverse wildly inaccurate.

</details>

---

**Question 3:** OLS gives the EXACT optimal solution in one step. Why would anyone use gradient descent instead?

<details>
<summary>▶ Click to reveal answer</summary>

OLS is perfect when feasible, but it breaks down in several common scenarios:

1. **Large p**: inverting XᵀX costs O(p³). For p=100,000 features (common in NLP), this is 10¹⁵ operations — computationally infeasible.
2. **Streaming/online data**: OLS requires seeing all data at once. GD can update incrementally (stochastic GD).
3. **Non-convex models**: Neural networks have non-quadratic losses — there is no closed-form solution. Only GD works.
4. **Regularised variants**: While ridge has a closed form, elastic net and lasso do not (lasso's L1 term is non-differentiable).
5. **Memory constraints**: X itself (n × p) might not fit in RAM. GD with mini-batches avoids loading all data.

Rule of thumb: use OLS for p < 1000–10,000. Use GD (or conjugate gradients) for larger problems.

</details>

---

**Question 4:** The normal equations are XᵀXθ = Xᵀy. If you have n=10,000,000 rows and p=10 features, what size is XᵀX? Is storing it feasible?

<details>
<summary>▶ Click to reveal answer</summary>

- XᵀX has shape **(p × p) = (10 × 10)** — only 100 numbers, regardless of n!
- Memory: 100 × 8 bytes (float64) = **800 bytes** — trivially small
- X itself is (10,000,000 × 10) = 100M entries = **~800 MB** — fits in RAM on modern hardware

So for this problem, OLS is perfectly feasible:
1. Compute XᵀX in one pass: O(np) = O(10⁸) operations
2. Compute Xᵀy in one pass: O(np) = O(10⁸) operations
3. Invert XᵀX: O(p³) = O(1000) operations — negligible!

The key insight: **the size of XᵀX depends only on p, not n**. For small p, OLS scales perfectly to arbitrarily large n.

Contrast with p=10,000: XᵀX is (10,000 × 10,000) = 800 MB just to store, and O(10¹²) to invert.

</details>